In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib
import os

from math import sqrt
SPINE_COLOR = 'gray'
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib as mpl

# Global style
mpl.rcParams.update({
    # Fonts
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",  # Computer Modern math

    # Text sizes
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,

    # Axes and ticks
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,

    # Lines
    "lines.linewidth": 1.8,

    # Figure export
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})

df = pd.read_csv("/Users/jonathan/compiled.csv")
df_duration = pd.read_csv("/Users/jonathan/duration_compiled.csv")

EVAL = "/Users/jonathan/eval_out"
os.makedirs("reefformer_plots", exist_ok=True)
df = pd.read_csv(f"{EVAL}/sweep_loss_weights/compiled.csv")
df_duration = pd.read_csv(f"{EVAL}/duration_compiled.csv")
df_vit = pd.read_csv(f"{EVAL}/vitsize/compiled.csv")


def _optional_csv(name):
    path = f"{EVAL}/{name}"
    if not os.path.isfile(path):
        print(f"not in eval_out yet: {path}")
        return None
    return pd.read_csv(path)


df_res = _optional_csv("hires/compiled.csv")
df_quality = _optional_csv("data_quality/compiled.csv")
df_size = _optional_csv("data_size/compiled.csv")

df39 = pd.read_csv(f"{EVAL}/coralscapes39.csv")
if "split" in df39.columns:
    df39 = df39[df39["split"] == "val"]
df39["resolution"] = df39["resolution"].astype(int)
df39["k"] = df39["k"].astype(int)
for col in ("accuracy", "miou"):
    df39[col] = pd.to_numeric(df39[col], errors="coerce")
# 39-class validation probe, 50 labeled pixels per class.
df_k50 = df39[df39["k"] == 50].copy()


def k50_scores(run, checkpoint, resolution):
    """Validation accuracy and mIoU for one 39-class k=50 cell, or NaN."""
    hit = df_k50[
        (df_k50["run"] == run)
        & (df_k50["checkpoint"] == checkpoint)
        & (df_k50["resolution"] == int(resolution))
    ]
    if hit.empty:
        return np.nan, np.nan
    return float(hit["accuracy"].iloc[0]), float(hit["miou"].iloc[0])


def duration_total_steps(row):
    """Peak epochN -> (N+1)*25k. Cooldown run ..._cd{S} -> S+25k."""
    if row["sweep"] == "duration_peak":
        epoch = int(str(row["checkpoint"]).removeprefix("epoch"))
        return (epoch + 1) * 25000
    if row["sweep"] == "duration_cooldown":
        source = int(str(row["run"]).rsplit("_cd", 1)[1])
        return source + 25000
    return np.nan




In [ ]:
df39 = pd.read_csv("/Users/jonathan/eval_out/coralscapes39.csv")
df39

In [ ]:
from matplotlib.lines import Line2D

KS = [5, 10, 20, 50]
ITERS_PER_EPOCH = 25000
COOLDOWN_ITERS = 25000
C_MAIN = plt.cm.coolwarm(0)  # duration run is pix 0.5


def _duration_steps(row):
    """Peak epochN -> (N+1)*25k. Cooldown run ..._cd{S} -> S+25k."""
    if row["sweep"] == "duration_peak":
        epoch = int(str(row["checkpoint"]).removeprefix("epoch"))
        return False, (epoch + 1) * ITERS_PER_EPOCH
    if row["sweep"] == "duration_cooldown":
        source = int(str(row["run"]).rsplit("_cd", 1)[1])
        return True, source + COOLDOWN_ITERS
    return None


df39 = pd.read_csv("/Users/jonathan/eval_out/coralscapes39.csv")
if "split" in df39.columns:
    df39 = df39[df39["split"] == "val"]
df39 = df39[df39["sweep"].isin(["duration_peak", "duration_cooldown"])].copy()
parsed = df39.apply(_duration_steps, axis=1)
df39["cooldown"] = [item[0] for item in parsed]
df39["total_steps"] = [item[1] for item in parsed]
df39["resolution"] = df39["resolution"].astype(int)
df39["k"] = df39["k"].astype(int)
for col in ("accuracy", "miou"):
    df39[col] = pd.to_numeric(df39[col], errors="coerce")


def _at(frame, step, col):
    hit = frame.loc[frame["total_steps"] == step, col]
    if hit.empty or not np.isfinite(hit.iloc[0]):
        return None
    return float(hit.iloc[0])


def _cooldown(axis, peak, cool, step, col, color):
    src = step - COOLDOWN_ITERS
    y0, y1 = _at(peak, src, col), _at(cool, step, col)
    if y0 is None or y1 is None:
        return
    axis.plot([src / 1000, step / 1000], [y0, y1], linestyle="--", color=color, linewidth=1.5)


def _legend(axis, entries):
    handles = [Line2D([0], [0], color=color, linestyle=ls, linewidth=1.5) for color, ls, _ in entries]
    axis.legend(
        handles,
        [label for _, _, label in entries],
        loc="best",
        fontsize=8,
        handlelength=3.4,
        framealpha=0.92,
    )


for resolution in [512]:
    fig, ax = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
    duration = df39[df39["resolution"] == resolution]
    peak_cool = [
        (C_MAIN, "-", "Peak LR (R=16)"),
        (C_MAIN, "--", "+25k cooldown"),
    ]

    for col_i, k in enumerate(KS):
        peak = duration[(duration["k"] == k) & ~duration["cooldown"]].sort_values("total_steps")
        cool = duration[(duration["k"] == k) & duration["cooldown"]].sort_values("total_steps")
        x = peak["total_steps"] / 1000

        ax[0, col_i].set_title(f"39-class mIoU, k={k}")
        ax[0, col_i].plot(x, peak["miou"], color=C_MAIN, marker="x")
        ax[0, col_i].set_ylabel("mIoU")
        ax[0, col_i].grid(True)

        ax[1, col_i].set_title(f"39-class accuracy, k={k}")
        ax[1, col_i].plot(x, peak["accuracy"], color=C_MAIN, marker="x")
        ax[1, col_i].set_xlabel("Training Iterations (k)")
        ax[1, col_i].set_ylabel("Accuracy")
        ax[1, col_i].grid(True)

        for step in cool["total_steps"].values:
            _cooldown(ax[0, col_i], peak, cool, step, "miou", C_MAIN)
            _cooldown(ax[1, col_i], peak, cool, step, "accuracy", C_MAIN)
        _legend(ax[0, col_i], peak_cool)
        _legend(ax[1, col_i], peak_cool)

    fig.tight_layout()
    plt.savefig("reefformer_plots/training_duration_coralscapes39.pdf")
    plt.show()

In [ ]:
#epoch = 2
resolution = 512

pix_weights = sorted(np.unique(df['pix_weight'].dropna().values))






for epoch in range(9, 10):
    fig, ax = plt.subplots(1, 2, figsize=(8.6,3.7))
    df_sub = df[np.logical_and(df['epoch']==epoch, df['resolution']==resolution)].copy()
    scores = df_k50[
        (df_k50["sweep"] == "lossw")
        & (df_k50["checkpoint"] == f"epoch{int(epoch)}")
        & (df_k50["resolution"] == resolution)
    ][["run", "accuracy", "miou"]]
    df_sub = df_sub.merge(scores, on="run", how="left").sort_values("pix_weight")
    c1 =  'black'
    c2 =  'tab:orange'
    l1 = "-"
    l2 = "--"
    
    ax1 = ax[1]
    ax1.set_title("CoralscapesV2 39-class, k=50")
    ax1.plot(df_sub['pix_weight'], df_sub['accuracy'], color=c1, linestyle=l1, marker='x')
    ax1.set_ylabel('Validation accuracy', color=c1)
    ax1.tick_params(axis='y', labelcolor=c1)
    ax1.grid(axis='y',color=c1, alpha=0.5, linestyle=l1)
    
    ax2 = ax1.twinx()  
    ax2.plot(df_sub['pix_weight'], df_sub['miou'], color=c2, marker='x', linestyle=l2)
    ax2.tick_params(axis='y', labelcolor=c2)
    ax2.set_ylabel('Validation mIoU', color=c2)
    ax1.set_xlabel('Per Image $\\longleftarrow$ Loss Weight $\\longrightarrow$ Per Pixel')
    ax2.grid(axis='y',color=c2, alpha=0.5, linestyle=l2)

    
    ax1.set_xlim(0, 1)
    ax2.set_xlim(0, 1)


    tax_living = [
        f"coralnet_tax_{rank}_acc_living"
        for rank in ("kingdom", "phylum", "class", "order", "family", "genus")
    ]
    ax1 = ax[0]
    ax1.set_title("Pacific labeled corals genus accuracy (living)")
    ax1.plot(df_sub['pix_weight'], df_sub['pacific_tax_genus_acc_living'], color=c1, linestyle=l1, marker='x')
    ax1.set_ylabel('Accuracy', color=c1)
    ax1.tick_params(axis='y', labelcolor=c1)
    
    ax2 = ax1.twinx()  
    ax2.plot(df_sub['pix_weight'], df_sub[tax_living].mean(axis=1), color=c2, marker='x', linestyle=l2)

    
    ax2.tick_params(axis='y', labelcolor=c2)
    ax2.set_ylabel('Mean taxonomic accuracy (living)', color=c2)
    ax1.set_xlabel('Per Image $\\longleftarrow$ Loss Weight $\\longrightarrow$ Per Pixel')
    ax2.grid(axis='y',color=c2, alpha=0.5, linestyle=l2)
    ax1.grid(axis='y',color=c1, alpha=0.5, linestyle=l1)
    ax1.set_xlim(0, 1)
    ax2.set_xlim(0, 1)
 
    plt.tight_layout()
    plt.savefig("reefformer_plots/loss_weight_final.pdf")

In [ ]:
# Training images at step 150000 and 512 px.
# exp1 and exp2 train on CoralNet only. Their full set is exp1_drop0, shared by both.
# exp3 drops images at random across the full mix. exp4 drops whole CoralNet
# sources and whole other datasets, most coral genera first, until the image
# count removed is closest to 10/20/50%.
# The baseline and the sparsified run both train on the full mix, 447,411 images.
# They are markers. The diamond is that mix with every dense mask sparsified
# (cbm_dq_sparse).
import json

FINAL_STEPS = 150_000
RESOLUTION = 512
# CoralNet-only sweeps. Nothing removed is exp1_drop0, shared by exp1 and exp2.
CORALNET_ONLY = {"exp1", "exp2"}
EXPERIMENTS = [
    ("exp1", "CoralNet images, random", "tab:blue"),
    ("exp2", "CoralNet sources, most coral genera", "tab:orange"),
    ("exp3", "All images, random", "tab:green"),
    ("exp4", "Sources, most coral genera", "tab:red"),
]
LEFT_ROWS = [
    ("pacific_genus", "Pacific labeled corals genus accuracy (living)"),
    ("class_miou", "Class mIoU"),
    ("bleached_f1", "Bleached F1"),
    ("anthro_f1", "Anthropogenic F1"),
    ("binary_f1", "Binary F1"),
]
RIGHT_ROWS = [
    ("k50_accuracy", "39-class accuracy, k=50"),
    ("k50_miou", "39-class mIoU, k=50"),
    ("cs_bleached_f1", "Bleached F1"),
    ("cs_anthro_f1", "Anthropogenic F1"),
    ("cs_binary_f1", "Binary F1"),
]
BASELINE_LABEL = "Full training mix"
FULL_MIX_IMAGES = 447_411
SPARSE_LABEL = "All dense data sparsified"
DATA_SIZE_PATH = "reefformer_plots/data_size.pdf"


def _finite(value):
    value = pd.to_numeric(value, errors="coerce")
    if getattr(value, "ndim", 0) != 0:
        return np.nan
    value = float(value)
    return value if np.isfinite(value) else np.nan


def _column(row, column):
    if row is None or column not in row.index:
        return np.nan
    return _finite(row[column])


def _coralnet_score(row, kind):
    """One CoralNet metric from a compiled eval row."""
    if row is None:
        return np.nan
    if kind == "class_miou":
        return _column(row, "coralnet_class_miou")
    if kind == "bleached_f1":
        return _column(row, "coralnet_bleached_f1")
    if kind == "anthro_f1":
        return _column(row, "coralnet_anthropogenic_f1")
    if kind == "binary_f1":
        return _column(row, "coralnet_binary_macro_f1")
    if kind == "pacific_genus":
        return _column(row, "pacific_tax_genus_acc_living")
    raise ValueError(kind)


def _match(frame, **equals):
    if frame is None:
        return None
    hit = frame
    for column, value in equals.items():
        hit = hit[hit[column] == value]
    if hit.empty:
        return None
    return hit.iloc[0]


def _coralnet_total():
    """Every CoralNet training image. exp1_drop0 records that count directly."""
    if df_size is None:
        return np.nan
    drop0 = _match(df_size, profile="exp1_drop0", steps=FINAL_STEPS, resolution=RESOLUTION)
    if drop0 is None:
        hit = df_size[df_size["profile"] == "exp1_drop0"]
        drop0 = None if hit.empty else hit.iloc[0]
    kept = _column(drop0, "n_images_kept")
    if np.isfinite(kept):
        return kept
    sub = df_size[df_size["experiment"].isin(list(CORALNET_ONLY))]
    for _, row in sub.iterrows():
        kept = _column(row, "n_images_kept")
        dropped = _column(row, "n_images_dropped")
        if np.isfinite(kept) and np.isfinite(dropped):
            return kept + dropped
    return np.nan


def _training_images(row):
    """Images this run trains on. ``n_images_kept`` is already that count."""
    return _column(row, "n_images_kept")


def _full_mix_rows():
    """Baseline eval row, then the ViT-L sweep row when a score is missing there."""
    row = _match(df_size, profile="baseline", steps=FINAL_STEPS, resolution=RESOLUTION)
    vit = _match(df_vit, vit="vitl", steps=FINAL_STEPS, resolution=RESOLUTION)
    return row, vit


def _series(experiment):
    """(x, row, fallback row) for the full set and the 10/20/50% drops.

    exp1 and exp2 start at exp1_drop0. exp3 and exp4 are only their own drops.
    The full mix is a separate marker, not a vertex of these lines.
    """
    points = []
    if experiment in CORALNET_ONLY:
        anchor = _match(df_size, profile="exp1_drop0", steps=FINAL_STEPS, resolution=RESOLUTION)
        x = _column(anchor, "n_images_kept")
        if np.isfinite(x):
            points.append((x, anchor, None))
    if df_size is None:
        return points
    sub = df_size[
        (df_size["experiment"] == experiment)
        & (df_size["steps"] == FINAL_STEPS)
        & (df_size["resolution"] == RESOLUTION)
    ].copy()
    sub["fraction"] = pd.to_numeric(sub["fraction"], errors="coerce")
    sub = sub[sub["fraction"].isin([0.1, 0.2, 0.5])]
    for _, row in sub.iterrows():
        x = _training_images(row)
        if np.isfinite(x):
            points.append((x, row, None))
    return points


def _point_score(row, fallback, kind, scorer):
    score = scorer(row, kind)
    if np.isfinite(score) or fallback is None:
        return score
    return scorer(fallback, kind)


def _mean_f1(row, bleached_column, anthro_column):
    values = np.asarray(
        [_column(row, bleached_column), _column(row, anthro_column)], dtype=float
    )
    if not np.isfinite(values).all():
        return np.nan
    return float(values.mean())


def _coralscapes_score(row, kind):
    """Validation Coralscapes metric. k=50 comes from the 39-class probe table."""
    if kind == "k50_accuracy":
        if row is None:
            return np.nan
        accuracy, _ = k50_scores(row["run"], f"step{FINAL_STEPS}", RESOLUTION)
        return accuracy
    if kind == "k50_miou":
        if row is None:
            return np.nan
        _, miou = k50_scores(row["run"], f"step{FINAL_STEPS}", RESOLUTION)
        return miou
    if kind == "cs_bleached_f1":
        return _column(row, "coralscapes_val_bleached_f1")
    if kind == "cs_anthro_f1":
        return _column(row, "coralscapes_val_anthropogenic_f1")
    if kind == "cs_binary_f1":
        return _mean_f1(row, "coralscapes_val_bleached_f1", "coralscapes_val_anthropogenic_f1")
    raise ValueError(kind)


def _sparse_summary():
    path = os.path.join(
        EVAL, "data_quality", "cbm_dq_sparse", f"step{FINAL_STEPS}_res{RESOLUTION}", "summary.json"
    )
    if not os.path.isfile(path):
        return {}
    with open(path) as handle:
        summary = json.load(handle)
    return summary if isinstance(summary, dict) else {}


def _pooled_coralnet(kind):
    if kind == "pacific_genus":
        pooled = (_sparse_summary().get("pacific") or {}).get("pooled") or {}
        genus = (pooled.get("taxonomic") or {}).get("genus") or {}
        return _finite(genus.get("acc_living"))
    pooled = (_sparse_summary().get("coralnet_val") or {}).get("pooled") or {}
    if kind == "class_miou":
        return _finite(pooled.get("class_miou"))
    if kind == "bleached_f1":
        return _finite((pooled.get("bleached") or {}).get("f1"))
    if kind == "anthro_f1":
        return _finite((pooled.get("anthropogenic") or {}).get("f1"))
    if kind == "binary_f1":
        return _finite((pooled.get("binary") or {}).get("macro_f1"))
    raise ValueError(kind)


def _sparsified_coralnet(kind):
    """Full mix, with every dense mask reduced to sparse points."""
    row = _match(df_quality, profile="sparse", steps=FINAL_STEPS, resolution=RESOLUTION)
    score = _coralnet_score(row, kind)
    if not np.isfinite(score):
        score = _pooled_coralnet(kind)
    return score


def _sparsified_k50():
    accuracy, miou = k50_scores("cbm_dq_sparse", f"step{FINAL_STEPS}", RESOLUTION)
    if np.isfinite(accuracy) and np.isfinite(miou):
        return accuracy, miou
    sidecar = os.path.join(
        EVAL,
        "data_quality",
        "cbm_dq_sparse",
        f"step{FINAL_STEPS}_res{RESOLUTION}",
        "coralscapes39",
        "summary.json",
    )
    if not os.path.isfile(sidecar):
        return accuracy, miou
    with open(sidecar) as handle:
        probe = json.load(handle)
    block = ((probe.get("coralscapes39_val") or {}).get("by_k") or {}).get("50") or {}
    val = block.get("val") or {}
    if not np.isfinite(accuracy):
        accuracy = _finite(val.get("accuracy"))
    if not np.isfinite(miou):
        miou = _finite(val.get("miou"))
    return accuracy, miou


def _sparsified_coralscapes(kind):
    if kind == "k50_accuracy":
        accuracy, _ = _sparsified_k50()
        return accuracy
    if kind == "k50_miou":
        _, miou = _sparsified_k50()
        return miou
    row = _match(df_quality, profile="sparse", steps=FINAL_STEPS, resolution=RESOLUTION)
    return _coralscapes_score(row, kind)


if df_size is None:
    print("data_size/compiled.csv is not in eval_out yet; only finished anchors can be drawn")

drop0 = _match(df_size, profile="exp1_drop0", steps=FINAL_STEPS, resolution=RESOLUTION)
for kind, ylabel in LEFT_ROWS:
    if not np.isfinite(_coralnet_score(drop0, kind)):
        print(
            f"{ylabel}: exp1_drop0 at step {FINAL_STEPS}, res {RESOLUTION} has no CoralNet score yet, "
            "so exp1 and exp2 stop at the finished drops"
        )
for kind, ylabel in RIGHT_ROWS:
    if not np.isfinite(_coralscapes_score(drop0, kind)):
        print(
            f"{ylabel}: exp1_drop0 at step {FINAL_STEPS}, res {RESOLUTION} has no Coralscapes score yet, "
            "so exp1 and exp2 stop at the finished drops"
        )

series = {experiment: _series(experiment) for experiment, _label, _color in EXPERIMENTS}
columns = [
    (LEFT_ROWS, _coralnet_score, _sparsified_coralnet, "CoralNet"),
    (RIGHT_ROWS, _coralscapes_score, _sparsified_coralscapes, "Coralscapes"),
]

fig, ax = plt.subplots(nrows=5, ncols=2, figsize=(8.6, 14), sharex=True)
image_formatter = mpl.ticker.FuncFormatter(lambda value, _pos: f"{value:,.0f}")
for column, (rows, scorer, sparse_scorer, title) in enumerate(columns):
    ax[0, column].set_title(title)
    for row_index, (kind, ylabel) in enumerate(rows):
        axis = ax[row_index, column]
        for experiment, label, color in EXPERIMENTS:
            xs, ys = [], []
            for x, row, fallback in series[experiment]:
                y = _point_score(row, fallback, kind, scorer)
                if np.isfinite(x) and np.isfinite(y):
                    xs.append(x)
                    ys.append(y)
            if not xs:
                continue
            order = np.argsort(xs)
            axis.plot(
                np.asarray(xs)[order],
                np.asarray(ys, dtype=float)[order],
                color=color,
                marker="x",
                label=label,
            )
        baseline_score = _point_score(*_full_mix_rows(), kind, scorer)
        if np.isfinite(baseline_score):
            axis.plot(
                [FULL_MIX_IMAGES],
                [baseline_score],
                linestyle="None",
                marker="o",
                color="black",
                markersize=6,
                zorder=5,
                label=BASELINE_LABEL,
            )
        sparse_score = sparse_scorer(kind)
        if np.isfinite(sparse_score):
            axis.plot(
                [FULL_MIX_IMAGES],
                [sparse_score],
                linestyle="None",
                marker="D",
                color="black",
                markersize=6,
                zorder=5,
                label=SPARSE_LABEL,
            )
        elif column == 0:
            print(
                f"no CoralNet {ylabel} for the sparsified dense model under {EVAL}/data_quality"
            )
        else:
            print(
                f"no Coralscapes {ylabel} for the sparsified dense model under {EVAL}/data_quality"
            )
        axis.set_ylabel(ylabel)
        axis.grid(True, alpha=0.3)
        axis.xaxis.set_major_formatter(image_formatter)

for axis in ax[-1, :]:
    axis.set_xlabel("Training images")

legend_order = [label for _experiment, label, _color in EXPERIMENTS] + [
    BASELINE_LABEL,
    SPARSE_LABEL,
]
seen = {}
for axis in ax.ravel():
    for handle, label in zip(*axis.get_legend_handles_labels()):
        seen.setdefault(label, handle)
handles = [seen[label] for label in legend_order if label in seen]
labels = [label for label in legend_order if label in seen]
if handles:
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=min(len(handles), 2),
        fontsize=8,
        frameon=False,
    )

plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.savefig(DATA_SIZE_PATH)
plt.show()



In [ ]:
from matplotlib.lines import Line2D

TAX_RANKS = ["kingdom", "phylum", "class", "order", "family", "genus"]
#TAX_RANKS = [ "genus"]
BINARY_CONCEPTS = ["bleached", "anthropogenic"]
TAX_COLS = [f"coralnet_tax_{rank}_acc_living" for rank in TAX_RANKS]
MEAN_COLOR = "tab:green"
ANTHRO_COLOR = "tab:orange"


def _at(frame, step, col):
    hit = frame.loc[frame["total_steps"] == step, col]
    if hit.empty or not np.isfinite(hit.iloc[0]):
        return None
    return float(hit.iloc[0])


def _cooldown(axis, peak, cool, step, col, color):
    src = step - 25000
    y0, y1 = _at(peak, src, col), _at(cool, step, col)
    if y0 is None or y1 is None:
        return
    axis.plot([src / 1000, step / 1000], [y0, y1], linestyle="--", color=color, linewidth=1.5)


def _legend(axis, entries):
    handles = [Line2D([0], [0], color=color, linestyle=ls, linewidth=1.5) for color, ls, _ in entries]
    axis.legend(
        handles,
        [label for _, _, label in entries],
        loc="best",
        fontsize=8,
        handlelength=3.4,
        framealpha=0.92,
    )


for resolution in [512]:
    fig, ax = plt.subplots(2, 3, figsize=(15, 8))

    peak = df_duration[(df_duration["resolution"] == resolution) & ~df_duration["cooldown"]].sort_values("total_steps").copy()
    cool = df_duration[(df_duration["resolution"] == resolution) & df_duration["cooldown"]].sort_values("total_steps").copy()
    probe = df_k50[df_k50["sweep"].isin(["duration_peak", "duration_cooldown"])].copy()
    probe["total_steps"] = probe.apply(duration_total_steps, axis=1)
    probe = probe.rename(columns={"accuracy": "cs39_accuracy", "miou": "cs39_miou"})
    probe = probe[["run", "resolution", "total_steps", "cs39_accuracy", "cs39_miou"]]
    peak = peak.merge(probe, on=["run", "resolution", "total_steps"], how="left")
    cool = cool.merge(probe, on=["run", "resolution", "total_steps"], how="left")
    x = peak["total_steps"] / 1000
    c_main = plt.cm.coolwarm(0)

    for frame in (peak, cool):
        frame["_tax_acc"] = frame[TAX_COLS].mean(axis=1)
        frame["_cs_bin_f1"] = frame[[f"coralscapes_val_{c}_f1" for c in BINARY_CONCEPTS]].mean(axis=1)
        frame["_cn_bin_f1"] = frame[[f"coralnet_{c}_f1" for c in BINARY_CONCEPTS]].mean(axis=1)

    ax[0, 0].set_title("CoralscapesV2 39-class, k=50")
    ax[0, 0].plot(x, peak["cs39_miou"], color=c_main, marker='x')
    ax[0, 0].set_xlabel("Training Iterations (k)")
    ax[0, 0].set_ylabel("mIoU")
    ax[0, 0].grid(True)

    ax[0, 1].set_title("CoralscapesV2 39-class accuracy, k=50")
    ax[0, 1].plot(x, peak["cs39_accuracy"], color=c_main, marker='x')
    ax[0, 1].set_xlabel("Training Iterations (k)")
    ax[0, 1].set_ylabel("Accuracy")
    ax[0, 1].grid(True)

    ax[0, 2].set_title("CoralscapesV2 F1")
    ax[0, 2].plot(x, peak["coralscapes_bleached_f1"], color=c_main, marker='x')
    ax[0, 2].plot(x, peak["coralscapes_anthropogenic_f1"], color=ANTHRO_COLOR)
    ax[0, 2].plot(x, peak["_cs_bin_f1"], color=MEAN_COLOR)
    ax[0, 2].set_xlabel("Training Iterations (k)")
    ax[0, 2].set_ylabel("F1")
    ax[0, 2].grid(True)

    ax[1, 0].set_title("Pacific labeled corals genus accuracy (living)")
    ax[1, 0].plot(x, peak["pacific_tax_genus_acc_living"], color=c_main)
    ax[1, 0].set_xlabel("Training Iterations (k)")
    ax[1, 0].set_ylabel("Accuracy")
    ax[1, 0].grid(True)

    ax[1, 1].set_title("Mean taxonomic accuracy (living)")
    ax[1, 1].plot(x, peak["_tax_acc"], color=c_main)
    ax[1, 1].set_xlabel("Training Iterations (k)")
    ax[1, 1].set_ylabel("Accuracy")
    ax[1, 1].grid(True)

    ax[1, 2].set_title("CoralNet F1")
    ax[1, 2].plot(x, peak["coralnet_bleached_f1"], color=c_main)
    ax[1, 2].plot(x, peak["coralnet_anthropogenic_f1"], color=ANTHRO_COLOR)
    ax[1, 2].plot(x, peak["_cn_bin_f1"], color=MEAN_COLOR)
    ax[1, 2].set_xlabel("Training Iterations (k)")
    ax[1, 2].set_ylabel("F1")
    ax[1, 2].grid(True)

    for step in cool["total_steps"].values:
        _cooldown(ax[0, 0], peak, cool, step, "cs39_miou", c_main)
        _cooldown(ax[0, 1], peak, cool, step, "cs39_accuracy", c_main)
        _cooldown(ax[0, 2], peak, cool, step, "coralscapes_bleached_f1", c_main)
        _cooldown(ax[0, 2], peak, cool, step, "coralscapes_anthropogenic_f1", ANTHRO_COLOR)
        _cooldown(ax[0, 2], peak, cool, step, "_cs_bin_f1", MEAN_COLOR)
        _cooldown(ax[1, 0], peak, cool, step, "pacific_tax_genus_acc_living", c_main)
        _cooldown(ax[1, 1], peak, cool, step, "_tax_acc", c_main)
        _cooldown(ax[1, 2], peak, cool, step, "coralnet_bleached_f1", c_main)
        _cooldown(ax[1, 2], peak, cool, step, "coralnet_anthropogenic_f1", ANTHRO_COLOR)
        _cooldown(ax[1, 2], peak, cool, step, "_cn_bin_f1", MEAN_COLOR)

    peak_cool = [
        (c_main, "-", "Peak LR (R=16)"),
        (c_main, "--", "+25k cooldown"),
    ]
    for axis in (ax[0, 0], ax[0, 1], ax[1, 0], ax[1, 1]):
        _legend(axis, peak_cool)
    f1_entries = [
        (c_main, "-", "Bleached, peak LR"),
        (c_main, "--", "Bleached, +25k cooldown"),
        (ANTHRO_COLOR, "-", "Anthropogenic, peak LR"),
        (ANTHRO_COLOR, "--", "Anthropogenic, +25k cooldown"),
        (MEAN_COLOR, "-", "Mean binary F1, peak LR"),
        (MEAN_COLOR, "--", "Mean binary F1, +25k cooldown"),
    ]
    _legend(ax[0, 2], f1_entries)
    _legend(ax[1, 2], f1_entries)

    f1_lo = min(ax[0, 2].get_ylim()[0], ax[1, 2].get_ylim()[0])
    f1_hi = max(ax[0, 2].get_ylim()[1], ax[1, 2].get_ylim()[1])
    ax[0, 2].set_ylim(f1_lo, f1_hi)
    ax[1, 2].set_ylim(f1_lo, f1_hi)

    fig.tight_layout()
    plt.savefig("reefformer_plots/training_duration_ablation.pdf")
    plt.show()


In [ ]:

if df_res is None:
    print("hires/compiled.csv is not in eval_out yet")
else:
    probe = df_k50[df_k50["sweep"] == "hires"].copy()
    probe["finetune_steps"] = probe["checkpoint"].str.removeprefix("step").astype(int)
    probe = probe.rename(columns={"resolution": "pred_resolution", "miou": "cs39_miou"})

    for finetune_steps in [50000]:

        fig, ax = plt.subplots(ncols=2, nrows=1, figsize=(8.6,3.7))

        for train_res in [256, 384, 512, 768]:
            sub_df = df_res[np.logical_and(df_res['finetune_steps']==finetune_steps, df_res['train_resolution']==train_res)].copy()
            sub_df["finetune_steps"] = sub_df["finetune_steps"].astype(int)
            sub_df["pred_resolution"] = sub_df["pred_resolution"].astype(int)
            sub_df = sub_df.merge(
                probe[["run", "finetune_steps", "pred_resolution", "cs39_miou"]],
                on=["run", "finetune_steps", "pred_resolution"],
                how="left",
            )
            if train_res==784:
                print(sub_df)

            ax[0].plot(sub_df['pred_resolution'],sub_df['pacific_tax_genus_acc_living'], color=plt.cm.coolwarm((train_res-128)/640), label="Finetuned at "+str(train_res)+"px", marker='x')
            ax[0].set_xlabel("Eval Resolution")
            ax[0].set_ylabel("Accuracy")
            ax[0].set_title("Pacific labeled corals genus accuracy (living)")

            ax[1].plot(sub_df['pred_resolution'],sub_df['cs39_miou'], color=plt.cm.coolwarm((train_res-128)/640), label="Finetuned at "+str(train_res)+"px", marker='x')
            ax[1].set_xlabel("Eval Resolution")
            ax[1].set_ylabel("mIoU")
            ax[1].set_title("CoralscapesV2 39-class, k=50")

        ax[0].grid(True)
        ax[1].grid(True)

        ax[0].set_xticks([256, 384, 512, 768, 1024])
        ax[1].set_xticks([256, 384, 512, 768, 1024])
        ax[0].legend()

        plt.savefig("reefformer_plots/resolution_ablation.pdf")


    

In [ ]:
#Each cell is the mean GPU time over 300 forwards after 20 warmup iterations. The forward returns the 703 concept probabilities and does not run the class 1×1. The per-rank softmax in the real concept head is elementwise on that same tensor. ViT-H+ at 1024 is the long cell: a few hundred iterations there can take several minutes.
#gpu NVIDIA L40S mean of 300 iters

#model   res   ms/img    img/s  peak GB
def maybe_float(p):
    try:
        return float(p)
    except:
        return p
        
q = [[maybe_float(y.strip()) for y in x.split()] for x in """ViT-S   256      7.7   129.03      1.4
 ViT-S   512     12.4    80.58      4.9
 ViT-S   784     28.7    34.85     11.5
 ViT-S  1024     47.3    21.12     10.9
 ViT-B   256      7.8   127.83      1.0
 ViT-B   512     14.3    70.14      2.2
 ViT-B   784     32.2    31.05      4.5
 ViT-B  1024     54.7    18.28      7.3
 ViT-L   256     12.8    78.28      1.8
 ViT-L   512     21.7    46.13      3.1
 ViT-L   784     48.6    20.59      5.5
 ViT-L  1024     79.3    12.62      8.3
ViT-H+   256     17.3    57.73      3.9
ViT-H+   512     35.6    28.07      5.3
ViT-H+   784     76.1    13.14      7.7
ViT-H+  1024    128.2     7.80     10.7""".split("\n")]

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Inference speed lookup
fps_data = {
    (row[0], int(row[1])): float(row[3])
    for row in q
}

vitsizes = ["vits", "vitb", "vitl", "vithplus"]
vitsizes2 = ["ViT-S", "ViT-B", "ViT-L", "ViT-H+"]

# Marker size encodes model size
marker_sizes = {
    "ViT-S": 5,
    "ViT-B": 8,
    "ViT-L": 11,
    "ViT-H+": 15,
}

# Color encodes resolution
resolutions = [256, 512]
resolution_colors = {
    256: "tab:red",
    512: "tab:blue",
}

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5), sharex=True)

for resolution in resolutions:
    fps = []
    genus_acc = []
    coralscapes_miou = []
    sizes = []

    for vitsize, vitsize2 in zip(vitsizes, vitsizes2):
        sub_df = df_vit[
            (df_vit["vit"] == vitsize) &
            (df_vit["resolution"] == resolution)
        ].sort_values("steps")

        # Use final checkpoint
        row = sub_df.iloc[-1]

        fps.append(fps_data[(vitsize2, resolution)])
        genus_acc.append(row["pacific_tax_genus_acc_living"])
        _, miou = k50_scores(row["run"], f"step{int(row['steps'])}", resolution)
        coralscapes_miou.append(miou)
        sizes.append(marker_sizes[vitsize2])

    # Sort by FPS
    order = np.argsort(fps)
    fps = np.array(fps)[order]
    genus_acc = np.array(genus_acc)[order]
    coralscapes_miou = np.array(coralscapes_miou)[order]
    sizes = np.array(sizes)[order]

    color = resolution_colors[resolution]

    # Connect points
    ax[0].plot(
        fps, genus_acc,
        color=color,
        linewidth=1.5,
    )
    ax[1].plot(
        fps, coralscapes_miou,
        color=color,
        linewidth=1.5,
    )

    # Plot individual points
    for x, y1, y2, size in zip(
        fps, genus_acc, coralscapes_miou, sizes
    ):
        ax[0].plot(
            x, y1,
            marker="o",
            markersize=size,
            color=color,
            linestyle="none",
        )
        ax[1].plot(
            x, y2,
            marker="o",
            markersize=size,
            color=color,
            linestyle="none",
        )

# Titles and labels
ax[0].set_title("Pacific labeled corals genus accuracy (living)")
ax[0].set_ylabel("Accuracy")

ax[1].set_title("CoralscapesV2 39-class, k=50")
ax[1].set_ylabel("Validation mIoU")

for a in ax:
    a.set_xlabel("Inference Speed (FPS)")
    a.grid(True, alpha=0.3)

# Resolution legend (color)
resolution_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color=resolution_colors[r],
        linestyle="-",
        markersize=7,
        label=f"{r} px",
    )
    for r in resolutions
]

# ViT size legend (marker size)
size_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="gray",
        linestyle="none",
        markersize=marker_sizes[v],
        label=v,
    )
    for v in vitsizes2
]

# Combined legend inside left panel
ax[0].legend(
    handles=resolution_handles + size_handles,
    loc="lower left",
    frameon=True,
    framealpha=0.9,
    fontsize=10,
    ncol=3,
    columnspacing=1.0,
    handletextpad=0.5,
)

plt.tight_layout()
plt.savefig(
    "reefformer_plots/fps_ablation.pdf",
    bbox_inches="tight",
)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Inference speed lookup
fps_data = {
    (row[0], int(row[1])): float(row[3])
    for row in q
}

vitsizes = ["vits", "vitb", "vitl", "vithplus"]
vitsizes2 = ["ViT-S", "ViT-B", "ViT-L", "ViT-H+"]

# Marker size encodes model size
marker_sizes = {
    "ViT-S": 5,
    "ViT-B": 8,
    "ViT-L": 11,
    "ViT-H+": 15,
}

# Color encodes resolution
resolutions = [512]
resolution_colors = {
    512: "tab:blue",
}

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5), sharex=True)

for resolution in resolutions:
    fps = []
    genus_acc = []
    coralscapes_miou = []
    sizes = []

    for vitsize, vitsize2 in zip(vitsizes, vitsizes2):
        sub_df = df_vit[
            (df_vit["vit"] == vitsize) &
            (df_vit["resolution"] == resolution)
        ].sort_values("steps")

        # Use final checkpoint
        row = sub_df.iloc[-1]

        fps.append(fps_data[(vitsize2, resolution)])
        genus_acc.append(row["pacific_tax_genus_acc_living"])
        _, miou = k50_scores(row["run"], f"step{int(row['steps'])}", resolution)
        coralscapes_miou.append(miou)
        sizes.append(marker_sizes[vitsize2])

    # Sort by FPS
    order = np.argsort(fps)
    fps = np.array(fps)[order]
    genus_acc = np.array(genus_acc)[order]
    coralscapes_miou = np.array(coralscapes_miou)[order]
    sizes = np.array(sizes)[order]

    color = resolution_colors[resolution]

    # Connect points
    ax[0].plot(
        fps, genus_acc,
        color=color,
        linewidth=1.5,
    )
    ax[1].plot(
        fps, coralscapes_miou,
        color=color,
        linewidth=1.5,
    )

    # Plot individual points
    for x, y1, y2, size in zip(
        fps, genus_acc, coralscapes_miou, sizes
    ):
        ax[0].plot(
            x, y1,
            marker="o",
            markersize=size,
            color=color,
            linestyle="none",
        )
        ax[1].plot(
            x, y2,
            marker="o",
            markersize=size,
            color=color,
            linestyle="none",
        )

# Titles and labels
ax[0].set_title("Pacific labeled corals genus accuracy (living)")
ax[0].set_ylabel("Accuracy")

ax[1].set_title("CoralscapesV2 39-class, k=50")
ax[1].set_ylabel("Validation mIoU")

for a in ax:
    a.set_xlabel("Inference Speed (FPS)")
    a.grid(True, alpha=0.3)


# ViT size legend (marker size)
size_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="gray",
        linestyle="none",
        markersize=marker_sizes[v],
        label=v,
    )
    for v in vitsizes2
]

# Combined legend inside left panel
ax[0].legend(
    handles= size_handles,
    loc="lower left",
    frameon=True,
    framealpha=0.9,
    fontsize=10,
    ncol=2,
    columnspacing=1.0,
    handletextpad=0.5,
)

plt.tight_layout()
plt.savefig(
    "reefformer_plots/fps_ablation.pdf",
    bbox_inches="tight",
)
plt.show()